# 实体相似度 —— FB15k-237 嵌入空间探索

**任务定义**：探索 KGE 模型学到的实体嵌入空间。通过余弦相似度检索与查询实体最相似的邻居，验证嵌入是否捕捉了实体的语义相似性。

**与前三类任务的区别**：链接预测、关系预测、三元组分类都依赖具体的监督信号（排名、分类）。实体相似度不需要额外训练 —— 直接从预训练的 link prediction 模型中提取实体嵌入矩阵，在嵌入空间中计算距离和最近邻。

**核心问题**：
- 同一领域（如电影演员）的实体是否在嵌入空间中聚集？
- 高度数实体（枢纽节点）是否位于嵌入空间中心？
- 哪些实体对在嵌入空间中最相似？它们的关系是否符合直觉？

**数据集**：FB15k-237，包含电影、音乐、体育、奖项、地理等多个领域的实体。由于 Freebase MID 不具可读性，我们通过实体参与的关系域来推断其类型。

In [ ]:
import sys
import time
import tempfile
import shutil
from collections import Counter
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

_project_root = Path().resolve().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from kge.config.loader import from_yaml
from kge.datasets import load_dataset, KGDataModule
from kge.models import build_model
from kge.trainer import create_trainer
from core.utils import get_device

from kge.notebooks import (
    plot_training_curves,
    plot_triple_split_summary,
    plot_tsne_embeddings,
    plot_nearest_neighbors_table,
    parse_fb_relation_domain,
    format_relation_name,
    infer_entity_domain,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

cfg = from_yaml("config/link_prediction-baseline.yaml")
print(f"编码器: {cfg.model.encoder_name} | 嵌入维度: {cfg.model.params.get('embedding_dim', 100)}")
print(f"数据集: {cfg.dataset.name}")

## 数据加载

加载 FB15k-237 并查看基本统计。实体相似度分析需要了解数据集中的领域分布，以便后续选择有代表性的查询实体。

In [ ]:
dataset = load_dataset(cfg.dataset.name, cfg.dataset.root)
print(f"实体数量: {dataset.num_entities:,}")
print(f"关系数量: {dataset.num_relations:,}")
print(f"训练三元组: {len(dataset.train_triples):,}")

fig = plot_triple_split_summary(dataset)
plt.show()

# 预计算关系域映射（后续用于实体类型推断）
id2rel = dataset.get_id_to_relation
relation_domains = {}
for rid in range(dataset.num_relations):
    relation_domains[rid] = parse_fb_relation_domain(id2rel[rid])

# 关系域分布
domain_counts = Counter(relation_domains.values())
print(f"\n关系域分布（共 {len(domain_counts)} 个域）:")
for domain, count in domain_counts.most_common(12):
    print(f"  {domain:20s}: {count} 个关系")

## 阶段一：训练链接预测模型获取嵌入

使用 TransE 在 FB15k-237 上训练链接预测模型。训练完成后，实体嵌入矩阵 `encoder.entity_embedding.weight` 即为所有实体的向量表示。

In [ ]:
device = get_device(cfg.runtime.device)
print(f"设备: {device}")

data_module = KGDataModule(cfg, dataset)
model = build_model(cfg, dataset.num_entities, dataset.num_relations)
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

trainer = create_trainer(cfg, model, data_module, device)

checkpoint_dir = Path(tempfile.mkdtemp(prefix="kge_es_"))
torch.manual_seed(42)
t0 = time.perf_counter()
history = trainer.train(checkpoint_dir)
print(f"训练耗时: {time.perf_counter() - t0:.1f}s")

shutil.rmtree(checkpoint_dir, ignore_errors=True)

for k in sorted(history):
    if k.startswith("test_"):
        print(f"LP Test {k.replace('test_', '').upper()}: {history[k][0]:.4f}")

## 阶段二：提取并归一化实体嵌入

从训练好的 encoder 中提取实体嵌入矩阵并 L2 归一化。归一化后，向量内积等价于余弦相似度：

$$\text{cosine}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|} = \mathbf{u}_{norm} \cdot \mathbf{v}_{norm}$$

In [ ]:
entity_embeddings = model.encoder.entity_embedding.weight.detach().cpu()
print(f"实体嵌入矩阵形状: [{entity_embeddings.shape[0]}, {entity_embeddings.shape[1]}]")

entity_embeddings_norm = F.normalize(entity_embeddings, p=2, dim=1)

# 验证归一化
norms = entity_embeddings_norm.norm(dim=1)
print(f"归一化后范数范围: [{norms.min():.4f}, {norms.max():.4f}]")

## 阶段三：最近邻检索

选择来自不同 Freebase 领域的高频实体作为查询，检索它们在嵌入空间中的 Top-10 最近邻。

**查询实体选择策略**：在每个主要领域（film、music、sports、award、location）中，选择参与三元组数量最多（度数最高）的实体作为查询。这些“枢纽”实体通常在知识图谱中扮演核心角色。

**实体显示**：Freebase MID 本身不可读，我们通过统计实体参与的关系域来推断其类型（如 `[film]`、`[music]`），为每个实体添加领域标签。

In [ ]:
# 计算实体度数
entity_degree = torch.zeros(dataset.num_entities, dtype=torch.long)
for triple in dataset.train_triples:
    h, _, t = triple.tolist()
    entity_degree[h] += 1
    entity_degree[t] += 1

# 为每个实体推断领域类型
id2ent = dataset.id_to_entity
entity_domain = {}
for eid in range(dataset.num_entities):
    entity_domain[eid] = infer_entity_domain(eid, dataset.train_triples, relation_domains)

# 在每个主要领域中选择度数最高的实体
target_domains = ["film", "music", "sports", "award", "location", "organization", "people"]
query_entities = []
for domain in target_domains:
    candidates = [eid for eid, d in entity_domain.items() if d == domain]
    if not candidates:
        continue
    candidates.sort(key=lambda eid: entity_degree[eid].item(), reverse=True)
    query_entities.extend(candidates[:2])

print(f"查询实体（共 {len(query_entities)} 个）:")
for eid in query_entities:
    deg = entity_degree[eid].item()
    domain = entity_domain[eid]
    print(f"  {id2ent[eid][:25]:30s} [type: {domain:15s}] degree={deg}")

In [ ]:
# 最近邻搜索
top_k = 10
query_emb = entity_embeddings_norm[query_entities]  # (Q, D)
sim_matrix = query_emb @ entity_embeddings_norm.T     # (Q, N)

# 排除自身
for i, eid in enumerate(query_entities):
    sim_matrix[i, eid] = -1.0

top_sims, top_indices = torch.topk(sim_matrix, k=top_k, dim=1)

# 构建显示数据
query_names = []
neighbor_names_list = []
similarities_list = []

for i, eid in enumerate(query_entities):
    q_domain = entity_domain[eid]
    q_name = f"{id2ent[eid][:20]} [{q_domain}]"
    query_names.append(q_name)

    nbrs, sims = [], []
    for j in range(top_k):
        nid = top_indices[i, j].item()
        n_domain = entity_domain.get(nid, "unknown")
        n_name = f"{id2ent[nid][:20]} [{n_domain}]"
        nbrs.append(n_name)
        sims.append(top_sims[i, j].item())
    neighbor_names_list.append(nbrs)
    similarities_list.append(sims)

fig = plot_nearest_neighbors_table(query_names, neighbor_names_list, similarities_list, top_k=top_k)
plt.show()

# 统计邻居领域匹配率
match_count = 0
total = 0
for i, eid in enumerate(query_entities):
    q_domain = entity_domain[eid]
    for j in range(top_k):
        nid = top_indices[i, j].item()
        if entity_domain.get(nid) == q_domain:
            match_count += 1
        total += 1
print(f"\n邻居领域匹配率: {match_count}/{total} = {match_count/total:.1%}")
print("(同一领域的实体是否倾向于在嵌入空间中彼此靠近)")

## 阶段四：t-SNE 可视化（按领域着色）

采样 3000 个实体，用 t-SNE 降到二维空间，按推断的领域类型着色。如果嵌入质量良好，相同领域的实体应形成可辨识的聚类。

In [ ]:
sample_size = 3000
rng = np.random.RandomState(42)
sample_indices = rng.choice(dataset.num_entities, size=sample_size, replace=False)

sample_embs = entity_embeddings[sample_indices]
sample_domains = np.array([entity_domain[eid] for eid in sample_indices])

# 将少数域合并为 "other"
domain_count = Counter(sample_domains)
major_domains = {d for d, c in domain_count.items() if c >= 30}
sample_domains_mapped = np.array([
    d if d in major_domains else "other" for d in sample_domains
])

# 构建标签映射
unique_domains = sorted(set(sample_domains_mapped))
domain_to_id = {d: i for i, d in enumerate(unique_domains)}
domain_labels = np.array([domain_to_id[d] for d in sample_domains_mapped])
label_names = {i: d for i, d in enumerate(unique_domains)}

fig = plot_tsne_embeddings(
    sample_embs, domain_labels,
    title="FB15k-237 实体嵌入 t-SNE 可视化（按领域着色）",
    color_map="tab10",
    label_names=label_names,
)
plt.show()

print("各域实体数（采样）:")
for d in unique_domains:
    count = (sample_domains_mapped == d).sum()
    print(f"  {d:20s}: {count}")

## 阶段五：t-SNE 可视化（按度数着色）

换一个视角：用实体度数（参与三元组的总次数）着色。高度数实体（知识图谱中的“枢纽”节点，如热门演员、大城市）往往被模型放置在嵌入空间的中心区域，与许多其他实体都相近。

In [ ]:
sample_degrees = entity_degree[sample_indices].numpy()

fig = plot_tsne_embeddings(
    sample_embs, sample_degrees,
    title="FB15k-237 实体嵌入 t-SNE 可视化（按度数着色）",
    color_map="plasma",
)
plt.show()

print(f"度数范围: [{sample_degrees.min()}, {sample_degrees.max()}]")
print(f"度数中位数: {np.median(sample_degrees):.0f}")

# 检查度数-相似度关系
high_deg = sample_indices[sample_degrees >= np.percentile(sample_degrees, 90)]
low_deg = sample_indices[sample_degrees <= np.percentile(sample_degrees, 10)]
print(f"\n高度数实体 (top 10%): {len(high_deg)} 个")
print(f"低度数实体 (bottom 10%): {len(low_deg)} 个")

## 总结

本 notebook 演示了如何从预训练的 KGE 模型中提取和分析实体嵌入空间：

1. **嵌入提取**：`encoder.entity_embedding.weight` 直接提供所有实体的向量表示，无需额外训练
2. **最近邻检索**：
   - 余弦相似度是一个有效的实体相似度度量，邻居领域匹配率反映了嵌入的语义一致性
   - 相同领域的实体（如都是电影演员）倾向于在嵌入空间中彼此靠近
   - 跨领域的相似实体对可能揭示了知识图谱中隐含的类比关系
3. **t-SNE 可视化**：
   - 按领域着色：可以观察到大致的领域聚类，但不同领域之间存在模糊边界（反映了跨领域的实体关联）
   - 按度数着色：高度数实体（枢纽节点）倾向于集中在嵌入空间中心，低度数实体散布在边缘
4. **TransE 嵌入的性质**：由于 TransE 的平移假设（h + r ≈ t），具有相似关系模式的实体自然会获得相似的嵌入，这是最近邻检索效果的基础

**局限与改进**：
- Freebase MID 不可读，实体类型推断依赖于关系参与模式，可能存在误判
- TransE 对 1-N 关系建模能力有限，影响嵌入质量；可尝试 RotatE 或 PairRE
- t-SNE 仅保留局部结构，聚类间距不能直接解释为原始空间的距离